In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
# create Spark session in local mode
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.01"

spark = (
    SparkSession.builder \
    .appName("SparkUI_demo") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()
)

print("Spark UI is availble at http://localhost:4040")

c:\Users\patil\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark UI is availble at http://localhost:4040


In [3]:
df = spark.read.\
    option('header', 'true').\
    parquet(r"C:\Users\patil\OneDrive\Documents\Data Engineering\Spark and Pyspark\DataTalks_club\fhvhv_tripdata_2026-01.parquet")

In [4]:
df.show(5)

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+------------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|cbd_congestion_fee|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+-------

In [ ]:

# from pyspark import SparkFiles

# url = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2023-01.parquet"
# spark.sparkContext.addFile(url) # add the file to Spark context

# file_name = url.split("/")[-1]
# file_path = "file://" + SparkFiles.get(file_name)

# # Read the file into a PySpark DataFrame
# df = spark.read.csv(file_path, header=True, inferSchema=True)

In [ ]:
# df = df.repartition(24)
# df.write.parquet(r"C:/Users/patil/OneDrive/Documents/Data Engineering/Spark and Pyspark/DataTalks_club/")

In [ ]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [9]:
from pyspark.sql import functions as F

df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2026-01-01|  2026-01-01|         262|          79|
| 2026-01-01|  2026-01-01|         195|          52|
| 2026-01-01|  2026-01-01|          25|         181|
| 2026-01-01|  2026-01-01|         141|         226|
| 2026-01-01|  2026-01-01|         226|         226|
| 2026-01-01|  2026-01-01|         260|          82|
| 2026-01-01|  2026-01-01|          82|          95|
| 2026-01-01|  2026-01-01|         144|         246|
| 2026-01-01|  2026-01-01|         162|         113|
| 2026-01-01|  2026-01-01|         148|         265|
| 2026-01-01|  2026-01-01|         226|         255|
| 2026-01-01|  2026-01-01|         255|          97|
| 2026-01-01|  2026-01-01|          65|         112|
| 2026-01-01|  2026-01-01|         169|         167|
| 2026-01-01|  2026-01-01|         167|          47|
| 2026-01-01|  2026-01-01|          78|       

In [10]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [11]:
crazy_stuff('B02884')

's/b44'

In [13]:

from pyspark.sql import types
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

c:\Users\patil\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\patil\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [15]:
# using udf
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/d4c| 2026-01-01|  2026-01-01|         262|          79|
|  e/d4c| 2026-01-01|  2026-01-01|         195|          52|
|  e/d4c| 2026-01-01|  2026-01-01|          25|         181|
|  e/d4e| 2026-01-01|  2026-01-01|         141|         226|
|  e/d4e| 2026-01-01|  2026-01-01|         226|         226|
|  e/d4e| 2026-01-01|  2026-01-01|         260|          82|
|  e/d4e| 2026-01-01|  2026-01-01|          82|          95|
|  e/d4e| 2026-01-01|  2026-01-01|         144|         246|
|  e/d4c| 2026-01-01|  2026-01-01|         162|         113|
|  e/d4e| 2026-01-01|  2026-01-01|         148|         265|
|  e/d4e| 2026-01-01|  2026-01-01|         226|         255|
|  e/d4e| 2026-01-01|  2026-01-01|         255|          97|
|  e/d4e| 2026-01-01|  2026-01-01|          65|         112|
|  e/d4e| 2026-01-01|  2

In [7]:
df.select('pickup_datetime','dropoff_datetime','PULocationID','DOLocationID').filter(df.hvfhs_license_num=='HV0003')

DataFrame[pickup_datetime: timestamp_ntz, dropoff_datetime: timestamp_ntz, PULocationID: int, DOLocationID: int]